# Student AI MVP on Colab

最終更新日時: 2026-07-09 10:32:45 +09:00

一次方程式を学習する生徒AIをColabで実行するためのノートブックです。GitHubからこのリポジトリをcloneして、mock確認、パラメータ編集、対話授業、4bit LLM実行、ログ確認まで行います。

## 1. Clone repository

`REPO_URL` は設定済みです。すでに `/content/student-ai` が存在する場合はcloneをスキップします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Optional Hugging Face login

Gemma系モデルやprivate/gated modelを使う場合に実行してください。

In [ ]:
# from huggingface_hub import login
# login()

## 4. Smoke test with mock model

モデルをダウンロードせず、状態管理・回答生成・ログ保存の経路を確認します。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=True)
result = sim.answer(
    student_id="S001",
    problem="2x + 3 = 11 を解いてください。",
)

print(result["answer"])

## 5. Model parameters

ここでモデルID、4bit量子化、生成パラメータを調整します。

- `MODEL_ID`: 使用するHugging FaceモデルID
- `load_in_4bit`: GPUメモリ節約。Colabでは基本 `True`
- `compute_dtype`: 計算精度。まず `bfloat16`、動かない場合は `float16`
- `max_new_tokens`: 生徒回答の最大長
- `temperature`: 高いほど回答が揺れる
- `top_p`: 低いほど候補語を絞る
- `repetition_penalty`: 同じ表現の繰り返しを抑える

In [ ]:
from src.config import GenerationConfig, ModelLoadConfig

# 使用するモデル。Gemma 3 4Bを使うなら "google/gemma-3-4b-it" などに変更します。
MODEL_ID = "Qwen/Qwen3-4B"

MODEL_LOAD_CONFIG = ModelLoadConfig(
    load_in_4bit=True,  # True: 4bit量子化でメモリ節約。Colabでは基本True
    bnb_4bit_quant_type="nf4",  # 通常はnf4
    bnb_4bit_use_double_quant=True,  # True: さらにメモリ節約
    compute_dtype="bfloat16",  # 動かないGPUではfloat16も試す
)

GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=256,  # 長い返答が必要なら増やす。短い生徒回答なら128-256
    temperature=0.7,  # 低いほど安定、高いほど生徒らしい揺れが出る
    top_p=0.9,  # 低いほど保守的。通常は0.8-0.95
    do_sample=True,  # True: 毎回少し変わる。False: 安定寄り
    repetition_penalty=1.05,  # 同じ語句の繰り返し抑制
)

## 6. Student parameters

ここで使う生徒IDと、生徒AIの状態パラメータを編集します。実行すると `data/students/{STUDENT_ID}.json` に保存されます。

知識状態は `0-100` です。心理・性格系は `very_low / low / medium / high / very_high` の5段階です。

- `score`: 一次方程式全体の総合理解度
- `can_solve_ax_plus_b_equals_c`: `ax + b = c` 型を解く力
- `can_transpose_terms`: 移項の理解
- `can_divide_by_coefficient`: 係数で割る理解
- `can_handle_negative_numbers`: 負の数への対応
- `can_handle_fractions`: 分数への対応
- `misconceptions`: 誤概念。誤答や迷いとして発話に反映される
- `learning_speed`: 授業後の知識スコアの伸びやすさ
- `self_efficacy`: 自己効力感。自信の強さ
- `question_tendency`: 質問しやすさ
- `motivation`: 学習意欲
- `big_five`: 性格傾向。発話量、丁寧さ、不安の出方などに影響

In [ ]:
from src.state_manager import StateManager

STUDENT_ID = "S002"

state_manager = StateManager("data/students")
student_state = state_manager.load_student(STUDENT_ID)

# ここを編集すると、生徒AIの反応と、授業後の伸び方が変わります。
student_state["knowledge_state"] = {
    "linear_equation": {
        "level": "low",  # scoreから自動更新されます
        "score": 25,  # 一次方程式全体の理解度。0=未理解、100=十分理解
        "can_solve_ax_plus_b_equals_c": 25,  # ax + b = c 型を解く力
        "can_transpose_terms": 5,  # 移項で符号を変える理解
        "can_divide_by_coefficient": 20,  # 3x=15 で3で割る理解
        "can_handle_negative_numbers": 5,  # マイナスを含む計算への対応
        "can_handle_fractions": 5,  # 分数を含む式への対応
    }
}
student_state["misconceptions"] = [
    "移項は項を反対側へ動かすだけで符号はそのままだと思っている",  # 誤概念1
    "3x = 15 のような式で、3を引けばよいと考えることがある",  # 誤概念2
]
student_state["learning_speed"] = "low"  # 授業後の知識スコアの伸びやすさ
student_state["big_five"] = {
    "openness": "medium",  # 新しい説明や別解を受け入れやすいか
    "conscientiousness": "low",  # 丁寧に途中式を書くか
    "extraversion": "medium",  # 発話量や反応の積極性
    "agreeableness": "high",  # 教師に素直に合わせる傾向
    "neuroticism": "high",  # 不安や焦りの出やすさ
}
student_state["self_efficacy"] = "low"  # 自分は解けると思える度合い
student_state["question_tendency"] = "high"  # わからない時に質問する傾向
student_state["motivation"] = "medium"  # 学習意欲

state_manager.save_student(student_state)
student_state

## 7. Create simulator

授業で使うシミュレーターを作ります。まず軽く試すなら `USE_MOCK_MODEL=True`、実LLMで授業するなら `False` にしてください。

In [ ]:
from src.student_ai import StudentAISimulator

USE_MOCK_MODEL = True  # 実LLMを使う場合は False

sim = StudentAISimulator(
    model_id=MODEL_ID,
    load_in_4bit=MODEL_LOAD_CONFIG.load_in_4bit,
    model_load_config=MODEL_LOAD_CONFIG,
    generation_config=GENERATION_CONFIG,
    use_mock_model=USE_MOCK_MODEL,
)

print(f"student_id={STUDENT_ID}, model={sim.agent.model_id}")

## 8. Interactive lesson

教師として発話を入力すると、生徒AIが返答します。終了するには `exit` または `quit` と入力してください。

In [ ]:
while True:
    teacher_message = input("教師> ").strip()
    if teacher_message.lower() in {"exit", "quit", "終了"}:
        print("授業を終了します。")
        break
    if not teacher_message:
        continue

    result = sim.respond(STUDENT_ID, teacher_message)
    print("生徒AI>", result["answer"])
    print()

## 9. One-shot problem

対話ループを使わず、1問だけ出す場合はこちらを使います。

In [ ]:
result = sim.answer(
    student_id=STUDENT_ID,
    problem="3x - 5 = 10 を解いてください。",
)

print(result["answer"])

## 10. Check logs

In [ ]:
from pathlib import Path

print(Path("data/logs/human_readable.md").read_text(encoding="utf-8")[-2000:])

## 11. Git update on Colab

GitHub側で更新したコードをColabへ取り込む場合は、次のセルを実行します。

In [ ]:
%cd /content/student-ai
!git status
!git pull

依存関係が変わった場合は、pull後に再インストールします。

In [ ]:
!pip install -q -r requirements.txt

pullで衝突したり、Colab上の作業内容を捨ててGitHubの最新版に戻したい場合は、次のセルでcloneし直します。必要な場合だけ実行してください。

In [ ]:
# 必要な場合だけ実行
# %cd /content
# !rm -rf student-ai
# !git clone {REPO_URL} /content/student-ai
# %cd /content/student-ai
# !pip install -q -r requirements.txt

## 12. Optional tests

標準テストはmock modelのみを使うため、LLMのダウンロードなしで実行できます。

In [ ]:
!python -m pytest